# CS-320 · YOLOv1 project · Hassaan (Data lead) — Part A: Penn-Fudan dataset
**Scope (from the project proposal):** dataset loading, YOLO-format conversion, splits, quality checks, statistics, split manifest, `data.yaml`.

| Step | What this notebook does | Output |
|---|---|---|
| 1 | Download + integrity-check Penn-Fudan (170 images) | `PennFudanPed/` |
| 2 | Inspect images, raw annotation text, instance masks | `sample_grid.png`, `mask_instances.png` |
| 3 | Convert PASCAL boxes → normalised YOLO boxes, cross-check against mask boxes | `labels/`, `conversion_log.txt` |
| 4 | 70/15/15 image-level split (seed 42; reuses the team manifest so Ayesha's/Sara's numbers stay comparable) | `split_manifest.csv` |
| 5 | 10 automated quality checks | `quality_report.txt` |
| 6 | Statistics + figures | `dataset_stats.csv`, `dataset_stats_plots.png`, `label_check_samples.png` |
| 7 | Dataset loader (with / without augmentation) for the YOLOv1 code | `yolotools.yolo_dataset` |
| 8 | Package for GitHub + a full zip for teammates | Drive |

## 0 · Setup

In [ ]:
import os, sys, json, shutil, random
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:                                   # Drive is used to keep results if Colab disconnects
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/yolo_project'
    os.makedirs(DRIVE, exist_ok=True)
except Exception:
    DRIVE = None
print('Drive folder:', DRIVE)

### Project modules
The cells below write the `yolotools` package to disk (same files that live in the GitHub repo under `yolotools/`). Run them once; nothing needs to be edited.

In [ ]:
!mkdir -p /content/cs320-yolov1/yolotools

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/__init__.py
"""CS-320 YOLOv1 project - Hassaan (Data lead) toolkit.

Modules
-------
boxes         box format conversions and IoU
penn_fudan    Penn-Fudan download, PASCAL->YOLO conversion, splits, dataset writer
quality       the 10 dataset quality checks + report
viz           dataset figures (sample grid, mask instances, label check, stats)
augment       box-aware augmentation (HSV, flip, scale+translate)
yolo_dataset  loader for the YOLO-format Penn-Fudan folder
voc2007       PASCAL VOC 2007 download + loader + YOLOv1 7x7xK target encoder
nms           NMS, class-aware NMS, YOLOv1 output decoding
metrics       VOC-style AP / mAP@IoU (07 11-point and all-point), P/R at threshold
evaluate      glue: run a YOLOv1-style model over a dataset and compute mAP
ablation      Ultralytics augmentation on/off ablation helpers
"""
__version__ = "1.0.0"

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/boxes.py
"""Box utilities (NumPy only).

Conventions
-----------
xyxy : (x1, y1, x2, y2)   continuous edge coordinates (x2 > x1)
xywh : (xc, yc, w, h)     centre + size
IoU is computed on continuous coordinates. Because IoU is invariant to scaling each
axis separately, it gives identical values on pixel or on normalised (0-1) boxes.
"""
import numpy as np


def xywh_to_xyxy(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], axis=1)


def xyxy_to_xywh(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2,
                     b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], axis=1)


def box_area(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)


def iou_matrix(a, b):
    """Pairwise IoU between a (N,4) and b (M,4) xyxy boxes -> (N,M)."""
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def iou_one_to_many(box, boxes):
    return iou_matrix(np.asarray(box).reshape(1, 4), boxes)[0]

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/penn_fudan.py
"""Penn-Fudan Pedestrian -> YOLO-format dataset (download, parse, convert, split, write).

Official PASCAL annotation boxes are 1-indexed INCLUSIVE pixel coordinates. They are converted to
continuous edges  (xmin-1, ymin-1, xmax, ymax)  and then to normalised YOLO `class xc yc w h`.
Every box is cross-checked against the box derived from the instance mask.
"""
import csv, json, os, re, shutil, urllib.request, zipfile
import numpy as np
from PIL import Image
from .boxes import iou_matrix, xyxy_to_xywh

URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
SEED = 42
RATIOS = (0.70, 0.15, 0.15)
EXPECTED = {"images": 170, "instances": 423}          # published Penn-Fudan totals
MIN_MASK_IOU_WARN = 0.70

_BOX_RE = re.compile(r"Bounding box for object\s+\d+.*?:\s*\((\d+),\s*(\d+)\)\s*-\s*\((\d+),\s*(\d+)\)")
_SIZE_RE = re.compile(r"Image size \(X x Y x C\)\s*:\s*(\d+)\s*x\s*(\d+)\s*x\s*(\d+)")


def download_penn_fudan(dest="/content", url=URL):
    """Download + unzip; returns the dataset root (folder containing PNGImages/)."""
    os.makedirs(dest, exist_ok=True)
    root = os.path.join(dest, "PennFudanPed")
    if os.path.isdir(os.path.join(root, "PNGImages")):
        return root
    zpath = os.path.join(dest, "PennFudanPed.zip")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
    return root


def parse_annotation(path):
    """-> dict(width, height, boxes=[(xmin,ymin,xmax,ymax)] in 1-indexed inclusive pixels)."""
    txt = open(path, encoding="latin-1").read()
    m = _SIZE_RE.search(txt)
    if not m:
        raise ValueError(f"no image size in {path}")
    boxes = [tuple(int(v) for v in g) for g in _BOX_RE.findall(txt)]
    return {"width": int(m.group(1)), "height": int(m.group(2)), "boxes": boxes}


def annotation_to_edges(b):
    xmin, ymin, xmax, ymax = b
    return (xmin - 1, ymin - 1, xmax, ymax)


def mask_instance_edges(mask_path):
    """Instance boxes from the mask (0 = background, 1..N = pedestrians) as continuous edges."""
    mask = np.array(Image.open(mask_path))
    out = []
    for i in np.unique(mask):
        if i == 0:
            continue
        ys, xs = np.where(mask == i)
        out.append((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    return np.array(out, dtype=np.float64).reshape(-1, 4)


def edges_to_yolo(edges, W, H):
    xc, yc, w, h = xyxy_to_xywh(np.asarray(edges, float).reshape(-1, 4)).T
    return np.stack([xc / W, yc / H, w / W, h / H], axis=1)


def yolo_to_edges(yolo, W, H):
    yolo = np.asarray(yolo, float).reshape(-1, 4)
    return np.stack([(yolo[:, 0] - yolo[:, 2] / 2) * W, (yolo[:, 1] - yolo[:, 3] / 2) * H,
                     (yolo[:, 0] + yolo[:, 2] / 2) * W, (yolo[:, 1] + yolo[:, 3] / 2) * H], axis=1)


def source_set(name):
    return "Penn" if name.startswith("PennPed") else "Fudan"


def build_records(root):
    """Parse + validate everything. Returns (records, issues, warnings)."""
    d_img, d_ann, d_msk = (os.path.join(root, d) for d in ("PNGImages", "Annotation", "PedMasks"))
    imgs = sorted(f for f in os.listdir(d_img) if f.lower().endswith(".png"))
    anns = {os.path.splitext(f)[0] for f in os.listdir(d_ann) if f.endswith(".txt")}
    msks = {f[:-len("_mask.png")] for f in os.listdir(d_msk) if f.endswith("_mask.png")}
    stems = {os.path.splitext(f)[0] for f in imgs}
    issues = {"unreadable_images": [], "missing_masks": [], "missing_annotations": [], "orphan_masks": [],
              "orphan_annotations": [], "size_mismatch": [], "invalid_boxes": [], "conversion_failures": []}
    warnings = []
    issues["missing_annotations"] = sorted(stems - anns)
    issues["missing_masks"] = sorted(stems - msks)
    issues["orphan_annotations"] = sorted(anns - stems)
    issues["orphan_masks"] = sorted(msks - stems)
    records = []
    for f in imgs:
        stem = os.path.splitext(f)[0]
        if stem in issues["missing_annotations"]:
            continue
        try:
            with Image.open(os.path.join(d_img, f)) as im:
                im.verify()
            with Image.open(os.path.join(d_img, f)) as im:
                W, H = im.size
        except Exception as e:                                  # noqa: BLE001
            issues["unreadable_images"].append(f"{f}: {e}")
            continue
        try:
            ann = parse_annotation(os.path.join(d_ann, stem + ".txt"))
        except Exception as e:                                  # noqa: BLE001
            issues["conversion_failures"].append(f"{f}: {e}")
            continue
        if (ann["width"], ann["height"]) != (W, H):
            issues["size_mismatch"].append(f"{f}: annotation {ann['width']}x{ann['height']} vs image {W}x{H}")
        edges = np.array([annotation_to_edges(b) for b in ann["boxes"]], float).reshape(-1, 4)
        bad = [i for i, e in enumerate(edges) if e[2] <= e[0] or e[3] <= e[1]]
        if bad:
            issues["invalid_boxes"].append(f"{f}: degenerate box index {bad}")
            edges = np.delete(edges, bad, axis=0)
        clipped = np.clip(edges, 0, [W, H, W, H])
        if len(edges) and not np.allclose(clipped, edges):
            warnings.append(f"{f}: box extended outside image and was clipped")
        edges = clipped
        miou = None
        if stem not in issues["missing_masks"]:
            mb = mask_instance_edges(os.path.join(d_msk, stem + "_mask.png"))
            if len(mb) != len(edges):
                warnings.append(f"{f}: {len(edges)} annotation boxes vs {len(mb)} mask instances")
            if len(mb) and len(edges):
                best = iou_matrix(edges, mb).max(axis=1)
                miou = float(best.min())
                if miou < MIN_MASK_IOU_WARN:
                    warnings.append(f"{f}: lowest annotation-vs-mask IoU {miou:.2f}")
        records.append({"filename": f, "source_set": source_set(f), "width": W, "height": H,
                        "edges": edges, "min_mask_iou": miou})
    return records, issues, warnings


def make_split(filenames, seed=SEED, ratios=RATIOS):
    """Image-level random split. Floor for train/val, remainder to test (170 -> 119/25/26)."""
    names = sorted(filenames)
    perm = np.random.RandomState(seed).permutation(len(names))
    n_tr = int(np.floor(ratios[0] * len(names) + 1e-9))   # epsilon: 0.7*170 == 118.99999...
    n_va = int(np.floor(ratios[1] * len(names) + 1e-9))
    split = {}
    for rank, i in enumerate(perm):
        split[names[i]] = "train" if rank < n_tr else ("val" if rank < n_tr + n_va else "test")
    return split


def load_manifest_split(path):
    with open(path, newline="") as f:
        return {r["filename"]: r["split"] for r in csv.DictReader(f)}


DATASET_README = """# PennFudanYOLO (Member 1 handover)
- Source: official Penn-Fudan zip ({url})
- Labels: YOLO detection format `0 xc yc w h` (normalised by each image's own size), class 0 = pedestrian
- Boxes: official PASCAL annotation boxes (1-indexed inclusive -> converted), cross-checked against mask instance boxes
- Split: image level 70/15/15, seed {seed}, see split_manifest.csv
- Images kept at original resolution (resizing is done by the training code; normalised labels are unaffected)
- Do NOT tune hyper-parameters on the test split
- Update `path:` in data.yaml to wherever this folder lives on your machine/Colab
"""


def write_dataset(root, out_dir, records, split, seed=SEED, issues=None, warnings=None):
    """Write images/, labels/, split_manifest.csv, dataset_stats.csv, data.yaml, README_dataset.md, conversion_log.txt."""
    if os.path.isdir(out_dir):
        for sub in ("images", "labels"):
            shutil.rmtree(os.path.join(out_dir, sub), ignore_errors=True)
    for s in ("train", "val", "test"):
        os.makedirs(os.path.join(out_dir, "images", s), exist_ok=True)
        os.makedirs(os.path.join(out_dir, "labels", s), exist_ok=True)
    rows = []
    for r in records:
        s = split[r["filename"]]
        shutil.copy2(os.path.join(root, "PNGImages", r["filename"]), os.path.join(out_dir, "images", s, r["filename"]))
        yolo = edges_to_yolo(r["edges"], r["width"], r["height"])
        with open(os.path.join(out_dir, "labels", s, os.path.splitext(r["filename"])[0] + ".txt"), "w") as f:
            for xc, yc, w, h in yolo:
                f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        rows.append({"filename": r["filename"], "split": s, "source_set": r["source_set"], "width": r["width"],
                     "height": r["height"], "n_pedestrians": len(yolo), "label_source": "annotation"})
    rows.sort(key=lambda x: x["filename"])
    with open(os.path.join(out_dir, "split_manifest.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    stats = [("Total", len(rows), sum(r["n_pedestrians"] for r in rows))]
    for s in ("train", "val", "test"):
        sel = [r for r in rows if r["split"] == s]
        stats.append((s.capitalize(), len(sel), sum(r["n_pedestrians"] for r in sel)))
    with open(os.path.join(out_dir, "dataset_stats.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["Metric", "Images", "Pedestrian instances"]); w.writerows(stats)
    with open(os.path.join(out_dir, "data.yaml"), "w") as f:
        f.write("# Auto-generated by Member 1 notebook (Penn-Fudan, single class)\n"
                f"path: {os.path.abspath(out_dir)}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                "nc: 1\nnames:\n  0: pedestrian\n")
    with open(os.path.join(out_dir, "README_dataset.md"), "w") as f:
        f.write(DATASET_README.format(url=URL, seed=seed))
    with open(os.path.join(out_dir, "conversion_log.txt"), "w") as f:
        f.write("ISSUES\n" + json.dumps(issues or {}, indent=2) + "\n\nWARNINGS\n" + "\n".join(warnings or []) + "\n")
    return rows, stats


def prepare(root, out_dir, manifest_path=None, seed=SEED):
    """Full pipeline. If manifest_path is given its split is reused (keeps team results comparable)."""
    records, issues, warnings = build_records(root)
    names = [r["filename"] for r in records]
    if manifest_path and os.path.exists(manifest_path):
        split = load_manifest_split(manifest_path)
        missing = set(names) - set(split)
        if missing:
            raise ValueError(f"manifest lacks {len(missing)} images, e.g. {sorted(missing)[:3]}")
        split = {n: split[n] for n in names}
    else:
        split = make_split(names, seed)
    rows, stats = write_dataset(root, out_dir, records, split, seed, issues, warnings)
    return records, split, issues, warnings, stats

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/quality.py
"""The 10 dataset quality checks -> quality_report.txt (PASS/FAIL per check)."""
import csv, os
import numpy as np
from PIL import Image
from .penn_fudan import parse_annotation, annotation_to_edges, yolo_to_edges

SPLITS = ("train", "val", "test")


def _read_labels(path):
    rows = []
    with open(path) as f:
        for ln in f:
            if ln.strip():
                rows.append(ln.split())
    return rows


def run_quality_checks(out_dir, root=None, expected_images=None, expected_instances=None, n_sample=40, seed=0):
    """Return list of (ok, number, title, detail). `root` (original Penn-Fudan) enables check 10."""
    res = []
    def add(i, title, problems):
        res.append((not problems, i, title, "all good" if not problems else f"{len(problems)} problem(s): {problems[:5]}"))

    imgs = {s: sorted(os.listdir(os.path.join(out_dir, "images", s))) for s in SPLITS}
    labs = {s: sorted(os.listdir(os.path.join(out_dir, "labels", s))) for s in SPLITS}
    # 1
    p = [f"{s}/{f}" for s in SPLITS for f in imgs[s] if os.path.splitext(f)[0] + ".txt" not in set(labs[s])]
    add(1, "Every image has a label file", p)
    # load all label rows once
    parsed = {}
    for s in SPLITS:
        for f in labs[s]:
            parsed[(s, f)] = _read_labels(os.path.join(out_dir, "labels", s, f))
    # 2
    p = [f"{s}/{f}" for (s, f), rows in parsed.items() for r in rows
         if len(r) != 5 or not all(_isfloat(v) for v in r)]
    add(2, "Label rows are valid YOLO (5 fields)", p)
    ok_rows = {k: [list(map(float, r)) for r in v if len(r) == 5 and all(_isfloat(x) for x in r)] for k, v in parsed.items()}
    # 3
    p = [f"{s}/{f}" for (s, f), rows in ok_rows.items() for r in rows if r[0] != 0 or r[0] != int(r[0])]
    add(3, "Class IDs valid (only 0)", p)
    # 4
    p = [f"{s}/{f}" for (s, f), rows in ok_rows.items() for r in rows
         if not (0 <= r[1] <= 1 and 0 <= r[2] <= 1 and 0 < r[3] <= 1 and 0 < r[4] <= 1
                 and r[1] - r[3] / 2 >= -1e-6 and r[1] + r[3] / 2 <= 1 + 1e-6
                 and r[2] - r[4] / 2 >= -1e-6 and r[2] + r[4] / 2 <= 1 + 1e-6)]
    add(4, "Normalised values within 0-1 (boxes inside image)", p)
    # 5
    p = [f"{s}/{f}" for (s, f), rows in ok_rows.items() for r in rows if r[3] <= 0 or r[4] <= 0]
    add(5, "Positive box width/height", p)
    # 6
    seen, p = {}, []
    for s in SPLITS:
        for f in imgs[s]:
            if f in seen:
                p.append(f"{f} in {seen[f]} and {s}")
            seen[f] = s
    add(6, "No image in more than one split", p)
    # 7
    p = [f"{s}: {len(imgs[s])} images vs {len(labs[s])} labels" for s in SPLITS if len(imgs[s]) != len(labs[s])]
    add(7, "Image count == label count per split", p)
    # 8
    p = []
    man = os.path.join(out_dir, "split_manifest.csv")
    if os.path.exists(man):
        rows = list(csv.DictReader(open(man)))
        for s in SPLITS:
            sel = [r for r in rows if r["split"] == s]
            n_inst = sum(len(v) for (ss, _), v in ok_rows.items() if ss == s)
            if len(sel) != len(imgs[s]):
                p.append(f"{s}: manifest {len(sel)} images vs folder {len(imgs[s])}")
            if sum(int(r["n_pedestrians"]) for r in sel) != n_inst:
                p.append(f"{s}: manifest instances != label rows ({n_inst})")
        if expected_images is not None and len(rows) != expected_images:
            p.append(f"total images {len(rows)} != expected {expected_images}")
        if expected_instances is not None and sum(int(r["n_pedestrians"]) for r in rows) != expected_instances:
            p.append("total instances != expected")
    else:
        p.append("split_manifest.csv missing")
    add(8, "Each split has the expected files / instance total", p)
    # 9
    p = []
    yml = os.path.join(out_dir, "data.yaml")
    if not os.path.exists(yml):
        p.append("data.yaml missing")
    else:
        cfg = {}
        for ln in open(yml):
            if ":" in ln and not ln.lstrip().startswith("#") and not ln.startswith(" "):
                k, v = ln.split(":", 1); cfg[k.strip()] = v.strip()
        base = cfg.get("path", out_dir)
        if not os.path.isdir(base):                      # data.yaml may point to Colab path; fall back to folder
            base = out_dir
        for k in ("train", "val", "test"):
            if not os.path.isdir(os.path.join(base, cfg.get(k, "?"))):
                p.append(f"{k} dir missing")
        if cfg.get("nc") != "1":
            p.append("nc != 1")
    add(9, "data.yaml points to valid directories", p)
    # 10
    p = []
    if root is None:
        res.append((True, 10, "Sampled saved labels match original boxes", "skipped (no original root given)"))
    else:
        rng = np.random.default_rng(seed)
        keys = sorted(ok_rows)
        for k in [keys[i] for i in rng.choice(len(keys), min(n_sample, len(keys)), replace=False)]:
            s, f = k
            stem = os.path.splitext(f)[0]
            with Image.open(os.path.join(out_dir, "images", s, stem + ".png")) as im:
                W, H = im.size
            ann = parse_annotation(os.path.join(root, "Annotation", stem + ".txt"))
            orig = np.array([annotation_to_edges(b) for b in ann["boxes"]], float).reshape(-1, 4)
            orig = np.clip(orig, 0, [W, H, W, H])
            saved = yolo_to_edges(np.array([r[1:] for r in ok_rows[k]]).reshape(-1, 4), W, H)
            if orig.shape != saved.shape or not np.allclose(orig, saved, atol=0.05):
                p.append(stem)
        add(10, "Sampled saved labels match original boxes", p)
    return res


def _isfloat(v):
    try:
        float(v); return True
    except ValueError:
        return False


def write_report(results, path):
    lines = [f"{'PASS' if ok else 'FAIL'}  {i}. {t}  -> {d}" for ok, i, t, d in results]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    return "\n".join(lines)

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/viz.py
"""Dataset figures: sample grid, mask instances, label check from SAVED labels, statistics plots."""
import csv, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from .penn_fudan import mask_instance_edges, yolo_to_edges
from .yolo_dataset import read_yolo_label

COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#008080", "#9a6324", "#000075"]


def _draw(ax, edges, color_offset=0):
    for k, (x1, y1, x2, y2) in enumerate(edges):
        c = COLORS[(k + color_offset) % len(COLORS)]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=2, ec=c, fc="none"))
        ax.text(x1, y1 - 2, "pedestrian", color="white", fontsize=7, weight="bold", bbox=dict(fc=c, ec="none", pad=1))


def plot_sample_grid(root, n=6, seed=0, save=None):
    files = sorted(os.listdir(os.path.join(root, "PNGImages")))
    pick = np.random.default_rng(seed).choice(len(files), n, replace=False)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4.5 * ((n + 1) // 2), 8))
    for ax, i in zip(axes.flat, pick):
        ax.imshow(Image.open(os.path.join(root, "PNGImages", files[i]))); ax.set_title(files[i], fontsize=9); ax.axis("off")
    plt.tight_layout()
    if save: fig.savefig(save, dpi=110, bbox_inches="tight")
    return fig


def plot_mask_instances(root, stem="FudanPed00013", save=None):
    img = Image.open(os.path.join(root, "PNGImages", stem + ".png"))
    mask = np.array(Image.open(os.path.join(root, "PedMasks", stem + "_mask.png")))
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    ax[0].imshow(img); ax[0].set_title("Image")
    ax[1].imshow(mask, cmap="gray"); ax[1].set_title("Raw mask (values 0,1,2,... look near-black)")
    ax[2].imshow(np.ma.masked_where(mask == 0, mask), cmap="tab10"); ax[2].set_title("Each ID = one pedestrian (mask-derived boxes)")
    _draw(ax[2], mask_instance_edges(os.path.join(root, "PedMasks", stem + "_mask.png")))
    for a in ax: a.axis("off")
    plt.tight_layout()
    if save: fig.savefig(save, dpi=110, bbox_inches="tight")
    return fig


def plot_label_check(out_dir, n=6, seed=0, save=None):
    """Draw boxes read back from the saved YOLO label files (what the models will actually see)."""
    items = [(s, f) for s in ("train", "val", "test") for f in sorted(os.listdir(os.path.join(out_dir, "images", s)))]
    pick = [items[i] for i in np.random.default_rng(seed).choice(len(items), n, replace=False)]
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4.5 * ((n + 1) // 2), 8))
    for ax, (s, f) in zip(axes.flat, pick):
        img = Image.open(os.path.join(out_dir, "images", s, f)); W, H = img.size
        _, xywh = read_yolo_label(os.path.join(out_dir, "labels", s, os.path.splitext(f)[0] + ".txt"))
        ax.imshow(img); _draw(ax, yolo_to_edges(xywh, W, H))
        ax.set_title(f"[{s}] {f} ({len(xywh)} boxes from saved labels)", fontsize=8); ax.axis("off")
    plt.tight_layout()
    if save: fig.savefig(save, dpi=110, bbox_inches="tight")
    return fig


def plot_statistics(out_dir, save=None):
    rows = list(csv.DictReader(open(os.path.join(out_dir, "split_manifest.csv"))))
    per_split = {s: [] for s in ("train", "val", "test")}
    hs, ws, ars = [], [], []
    for r in rows:
        _, xywh = read_yolo_label(os.path.join(out_dir, "labels", r["split"], os.path.splitext(r["filename"])[0] + ".txt"))
        per_split[r["split"]].append(len(xywh))
        if len(xywh):
            hs += list(xywh[:, 3]); ws += list(xywh[:, 2])
            ars += list((xywh[:, 2] * int(r["width"])) / (xywh[:, 3] * int(r["height"])))
    fig, ax = plt.subplots(1, 4, figsize=(19, 4))
    ax[0].bar(list(per_split), [len(v) for v in per_split.values()], color="#4363d8"); ax[0].set_title("Images per split")
    mx = max(max(v) for v in per_split.values() if v)
    for s, v in per_split.items():
        ax[1].hist(v, bins=np.arange(0.5, mx + 1.5), alpha=.55, label=s, density=True)
    ax[1].set_title("Pedestrians per image"); ax[1].legend()
    ax[2].hist(hs, bins=30, color="#3cb44b"); ax[2].set_title("Box height / image height")
    ax[3].hist(ars, bins=30, color="#f58231"); ax[3].set_title("Box aspect ratio (w/h, pixels)")
    plt.tight_layout()
    if save: fig.savefig(save, dpi=110, bbox_inches="tight")
    return fig, {"box_h_median": float(np.median(hs)), "box_w_median": float(np.median(ws)),
                 "aspect_median": float(np.median(ars)), "small_boxes_h<0.2": int(np.sum(np.array(hs) < 0.2))}

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/augment.py
"""Box-aware augmentation for YOLOv1-style loaders (PIL + NumPy).

Follows the YOLOv1 paper: random scale + translation up to 20 %, HSV exposure/saturation jitter up to
x1.5, plus horizontal flip. Boxes are normalised xyxy (0-1) and are transformed together with the image.
"""
import numpy as np
from PIL import Image


def hsv_jitter(img, rng, hue=0.05, sat=1.5, val=1.5):
    hsv = np.array(img.convert("HSV"), dtype=np.float32)
    fs = sat ** rng.uniform(-1, 1)
    fv = val ** rng.uniform(-1, 1)
    hsv[..., 0] = (hsv[..., 0] + rng.uniform(-hue, hue) * 255) % 256
    hsv[..., 1] = np.clip(hsv[..., 1] * fs, 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * fv, 0, 255)
    return Image.fromarray(hsv.astype(np.uint8), "HSV").convert("RGB")


def hflip(img, boxes):
    out = boxes.copy()
    out[:, [0, 2]] = 1.0 - boxes[:, [2, 0]]
    return img.transpose(Image.FLIP_LEFT_RIGHT), out


def scale_translate(img, boxes, rng, max_scale=0.2, max_shift=0.2, min_visible=0.3, min_size=0.01):
    """Return (img, boxes, keep_mask). Boxes losing <min_visible of their area or smaller than min_size are dropped."""
    W, H = img.size
    s = rng.uniform(1 - max_scale, 1 + max_scale)
    tx, ty = rng.uniform(-max_shift, max_shift, 2)
    # output(x', y') samples input at (x'-(0.5+t)W)/s + 0.5W
    coeffs = (1 / s, 0, 0.5 * W - (0.5 + tx) * W / s, 0, 1 / s, 0.5 * H - (0.5 + ty) * H / s)
    img = img.transform((W, H), Image.AFFINE, coeffs, resample=Image.BILINEAR, fillcolor=(127, 127, 127))
    nb = boxes.copy()
    nb[:, [0, 2]] = (boxes[:, [0, 2]] - 0.5) * s + 0.5 + tx
    nb[:, [1, 3]] = (boxes[:, [1, 3]] - 0.5) * s + 0.5 + ty
    area0 = np.clip(nb[:, 2] - nb[:, 0], 0, None) * np.clip(nb[:, 3] - nb[:, 1], 0, None)
    nb = np.clip(nb, 0.0, 1.0)
    area1 = (nb[:, 2] - nb[:, 0]) * (nb[:, 3] - nb[:, 1])
    keep = (area1 >= min_visible * np.maximum(area0, 1e-12)) & ((nb[:, 2] - nb[:, 0]) >= min_size) & ((nb[:, 3] - nb[:, 1]) >= min_size)
    return img, nb, keep


def random_augment(img, boxes, labels, rng=None, p_flip=0.5, p_hsv=1.0, p_affine=1.0,
                   flip=True, hsv=True, affine=True):
    """img: PIL RGB; boxes (N,4) normalised xyxy; labels (N,). Each op can be switched off for ablations."""
    rng = rng or np.random.default_rng()
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4).copy()
    labels = np.asarray(labels).copy()
    if hsv and rng.random() < p_hsv:
        img = hsv_jitter(img, rng)
    if flip and rng.random() < p_flip:
        img, boxes = hflip(img, boxes)
    if affine and rng.random() < p_affine:
        img, boxes, keep = scale_translate(img, boxes, rng)
        boxes, labels = boxes[keep], labels[keep]
    return img, boxes, labels

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/yolo_dataset.py
"""Loader for the YOLO-format Penn-Fudan folder (images/<split>, labels/<split>).

Interface matches the team's YOLOv1 code: __getitem__ -> (image CHW float tensor 0-1, boxes (N,4) tensor
[xc,yc,w,h] normalised) and `collate_fn` (variable number of boxes per image). Also exposes
`ground_truth(i)` and `image_tensor(i)` for evaluate.py.
"""
import os
import numpy as np
from PIL import Image
from .augment import random_augment
from .boxes import xywh_to_xyxy, xyxy_to_xywh

try:
    import torch
    from torch.utils.data import Dataset as _Base
except ImportError:                                   # lets the module be imported / tested without torch
    torch = None
    _Base = object


def read_yolo_label(path):
    if not os.path.exists(path):
        return np.zeros(0, int), np.zeros((0, 4))
    rows = [ln.split() for ln in open(path) if ln.strip()]
    if not rows:
        return np.zeros(0, int), np.zeros((0, 4))
    arr = np.array(rows, dtype=np.float64)
    return arr[:, 0].astype(int), arr[:, 1:5]


class YoloFolderDataset(_Base):
    def __init__(self, root, split="train", img_size=448, augment=False, aug_kwargs=None):
        self.root, self.split, self.img_size, self.augment = root, split, img_size, augment
        self.aug_kwargs = aug_kwargs or {}
        d = os.path.join(root, "images", split)
        self.files = sorted(f for f in os.listdir(d) if f.lower().endswith((".png", ".jpg", ".jpeg")))

    def __len__(self):
        return len(self.files)

    def _raw(self, i):
        f = self.files[i]
        img = Image.open(os.path.join(self.root, "images", self.split, f)).convert("RGB")
        labels, xywh = read_yolo_label(os.path.join(self.root, "labels", self.split, os.path.splitext(f)[0] + ".txt"))
        return img, labels, xywh

    def sample(self, i, rng=None):
        """numpy sample: (uint8 HxWx3 image, boxes_xywh (N,4) float32, labels)."""
        img, labels, xywh = self._raw(i)
        boxes = xywh_to_xyxy(xywh)
        if self.augment:
            img, boxes, labels = random_augment(img, boxes, labels, rng, **self.aug_kwargs)
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        return np.asarray(img), xyxy_to_xywh(boxes).astype(np.float32), labels

    def ground_truth(self, i):
        _, labels, xywh = self._raw(i)
        return {"boxes": xywh_to_xyxy(xywh), "labels": labels, "difficult": np.zeros(len(labels), bool)}

    def image_tensor(self, i):
        img, _, _ = self._raw(i)
        arr = np.asarray(img.resize((self.img_size, self.img_size), Image.BILINEAR))
        return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0

    def __getitem__(self, i):
        arr, boxes, _ = self.sample(i)
        return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0, torch.from_numpy(boxes.reshape(-1, 4))


def collate_fn(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]

In [ ]:
import sys
sys.path.insert(0, '/content/cs320-yolov1')
import yolotools
print('yolotools', yolotools.__version__, 'ready')

## 1 · Download and integrity check

In [ ]:
from yolotools import penn_fudan as pf
ROOT = pf.download_penn_fudan('/content')
for d in ['PNGImages', 'Annotation', 'PedMasks']:
    print(f"{d:10s}", len(os.listdir(f'{ROOT}/{d}')), 'files')
assert len(os.listdir(f'{ROOT}/PNGImages')) == pf.EXPECTED['images'] == 170

## 2 · Inspect the raw data
Penn-Fudan has two sources: `FudanPed*` (Fudan University campus) and `PennPed*` (University of Pennsylvania). The annotation text files use **1-indexed inclusive** pixel boxes; the masks encode every pedestrian with its own integer ID (0 = background).

In [ ]:
print(open(f'{ROOT}/Annotation/FudanPed00001.txt', encoding='latin-1').read())
from yolotools import viz
viz.plot_sample_grid(ROOT, n=6, save='/content/sample_grid.png'); plt.show()
viz.plot_mask_instances(ROOT, 'FudanPed00013', save='/content/mask_instances.png'); plt.show()

## 3 · Convert to YOLO format
For each image: parse the official annotation boxes `(xmin,ymin,xmax,ymax)` (1-indexed, inclusive) → continuous edges `(xmin-1, ymin-1, xmax, ymax)` → normalised `class xc yc w h` (class 0 = pedestrian).  
Every annotation box is **cross-checked against the box derived from the instance mask**; low agreement or a different count is logged as a warning.

**Split.** The team's existing `split_manifest.csv` (image-level 70/15/15, seed 42, 119/25/26 images) is reused when found, because Ayesha's YOLOv1 notebook already trains on it — regenerating would change the splits and make results incomparable. If it cannot be found, an identical-recipe split is generated with seed 42.

In [ ]:
# look for the team manifest (GitHub repo first, then a file uploaded next to the notebook)
!rm -rf /content/team_repo && git clone -q --depth 1 https://github.com/ayeshaalamexe/cs320-yolov1.git /content/team_repo 2>&1 | tail -1
cands = ['/content/team_repo/PennFudanYOLO/split_manifest.csv', '/content/split_manifest.csv']
MANIFEST = next((c for c in cands if os.path.exists(c)), None)
print('manifest:', MANIFEST or 'none found -> generating a new seed-42 split')

OUT = '/content/PennFudanYOLO'
records, split, issues, warnings, stats = pf.prepare(ROOT, OUT, manifest_path=MANIFEST, seed=SEED)
print(json.dumps(issues, indent=2))
print('warnings:', len(warnings)); [print(' -', w) for w in warnings[:15]]

In [ ]:
import pandas as pd
miou = pd.Series([r['min_mask_iou'] for r in records if r['min_mask_iou'] is not None])
print('lowest annotation-vs-mask IoU per image:\n', miou.describe().round(3).to_string())
pd.read_csv(f'{OUT}/dataset_stats.csv')

## 4 · Split manifest
`split_manifest.csv` lists every image with its split, source set (Fudan/Penn), size and pedestrian count. Splits are by **image**; note that near-identical frames from one scene can land in different splits — a known limitation of image-level splitting on this small dataset.

In [ ]:
man = pd.read_csv(f'{OUT}/split_manifest.csv')
print(man.groupby(['split', 'source_set']).size().unstack(fill_value=0))
assert man.filename.is_unique                       # no image twice
man.head()

## 5 · Quality checks (10 automated checks)

In [ ]:
from yolotools import quality
res = quality.run_quality_checks(OUT, ROOT, expected_images=pf.EXPECTED['images'], expected_instances=pf.EXPECTED['instances'])
print(quality.write_report(res, f'{OUT}/quality_report.txt'))
assert all(r[0] for r in res), 'a quality check FAILED - read the line above'

## 6 · Statistics and visual label check
The label-check figure draws boxes **read back from the saved label files**, i.e. what the models will actually be trained on.

In [ ]:
fig, extra = viz.plot_statistics(OUT, save=f'{OUT}/dataset_stats_plots.png'); plt.show()
print(extra)
viz.plot_label_check(OUT, n=6, seed=1, save=f'{OUT}/label_check_samples.png'); plt.show()
shutil.copy('/content/sample_grid.png', OUT); shutil.copy('/content/mask_instances.png', OUT)

## 7 · Dataset loader for the YOLOv1 code
`YoloFolderDataset` returns `(image, boxes)` exactly like the team's `PennFudanDataset` (image `3×448×448` in 0-1, boxes `[xc,yc,w,h]` normalised) so it plugs into the existing `collate_fn` and `encode_targets`. `augment=True` applies box-aware HSV jitter, horizontal flip and random scale/translate (YOLOv1-paper style) — this is the switch for the augmentation ablation on the custom YOLOv1.

In [ ]:
import torch
from torch.utils.data import DataLoader
from yolotools.yolo_dataset import YoloFolderDataset, collate_fn
plain = YoloFolderDataset(OUT, 'train', 448)
aug   = YoloFolderDataset(OUT, 'train', 448, augment=True)
loader = DataLoader(aug, batch_size=8, shuffle=True, collate_fn=collate_fn)
x, b = next(iter(loader)); print('batch', tuple(x.shape), '| boxes in img0', tuple(b[0].shape))

import matplotlib.patches as patches
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for k in range(5):
    for row, ds in enumerate((plain, aug)):
        arr, bx, _ = ds.sample(k * 7)
        ax = axes[row, k]; ax.imshow(arr); ax.axis('off'); ax.set_title(('plain' if row == 0 else 'augmented') + f' #{k*7}')
        for xc, yc, w, h in bx:
            ax.add_patch(patches.Rectangle(((xc - w/2) * 448, (yc - h/2) * 448), w * 448, h * 448, lw=2, ec='lime', fc='none'))
plt.tight_layout(); plt.show()

## 8 · Package for GitHub and for teammates

In [ ]:
GH = '/content/github_ready/PennFudanYOLO'           # small files only (images are re-creatable by this notebook)
shutil.rmtree('/content/github_ready', ignore_errors=True); os.makedirs(GH)
for f in ['split_manifest.csv', 'dataset_stats.csv', 'data.yaml', 'README_dataset.md', 'quality_report.txt', 'conversion_log.txt',
          'sample_grid.png', 'mask_instances.png', 'label_check_samples.png', 'dataset_stats_plots.png']:
    shutil.copy(f'{OUT}/{f}', GH)
shutil.copytree(f'{OUT}/labels', f'{GH}/labels')
shutil.make_archive('/content/PennFudanYOLO_full', 'zip', '/content', 'PennFudanYOLO')   # images + labels for Ayesha / Sara
if DRIVE:
    shutil.copy('/content/PennFudanYOLO_full.zip', DRIVE)
    shutil.copytree('/content/github_ready', f'{DRIVE}/github_ready', dirs_exist_ok=True)
    print('saved to Drive:', DRIVE)
print(os.path.getsize('/content/PennFudanYOLO_full.zip') // 1024, 'KB full zip')
print(open(f'{GH}/data.yaml').read())

**Done with Part A.** Copy `github_ready/PennFudanYOLO/*` into the repo's `PennFudanYOLO/` folder (see `docs/GITHUB_PUSH_GUIDE.md`). Share `PennFudanYOLO_full.zip` with teammates; in Colab they only need to unzip it and use `data.yaml`.